## 02 — EDA Bivariata

> **Analisi esplorativa bivariata del dataset pulito NBA.**  
> Obiettivo: quantificare le relazioni tra variabili, misurare il potere discriminante sul target W/L,  
> identificare multicollinearità e produrre ipotesi per la fase di feature engineering.
>
> **Input:** `data/2_processed/cleaned_nba_data_2000-01_2025-26.csv`  
> **Output:** heatmap correlazioni, ranking predittivo, trend temporali, analisi home/away, ipotesi per modellazione.

---

**Struttura del notebook**
1. Setup & caricamento dati
2. Correlazioni tra variabili numeriche
3. Feature vs Target (W/L)
4. Analisi Home vs Away
5. Analisi temporale
6. Analisi per team
7. Multicollinearità
8. Sintesi e ipotesi

### 1. Setup

In [ ]:
import logging
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve().parent.parent
sys.path.append(str(ROOT))

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import polars as pl
import polars.selectors as cs
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu, ttest_ind
from statsmodels.stats.outliers_influence import variance_inflation_factor

import itables.options as opt
from itables import init_notebook_mode, show

from src.loader import load_config

warnings.filterwarnings("ignore")

config = load_config()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    stream=sys.stdout,
    force=True,
)
log = logging.getLogger(__name__)

init_notebook_mode(all_interactive=True)
opt.warn_on_undocumented_option = False
opt.maxBytes   = 0
opt.pageLength = 15
opt.lengthMenu = [15, 25, 50, 100]
opt.scrollX    = True
opt.classes    = "display nowrap compact"
opt.style      = "width:100%; font-size:13px;"

# ── Plotting configuration ────────────────────────────────────────────────────
NBA_PALETTE  = ["#1d428a", "#c8102e", "#fdb927", "#007a33", "#552583", "#ce1141"]
ACCENT       = "#1d428a"
WIN_COLOR    = "#007a33"
LOSS_COLOR   = "#c8102e"
HOME_COLOR   = "#1d428a"
AWAY_COLOR   = "#fdb927"
NEUTRAL      = "#6c757d"
FIG_W_SINGLE = 10
FIG_H_SINGLE = 4
FIG_W_GRID   = 16
FIG_H_ROW    = 4
DPI          = 130

sns.set_theme(style="whitegrid", palette=NBA_PALETTE)
plt.rcParams.update({
    "figure.dpi"       : DPI,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "font.size"        : 11,
    "axes.titlesize"   : 13,
    "axes.labelsize"   : 11,
})

# ── Statistical thresholds ───────────────────────────────────────────────────
ALPHA         = 0.05
VIF_THRESHOLD = 5.0
HIGH_CORR     = 0.80

NOTEBOOK_VERSION = "1.0.0"
log.info(f"EDA Bivariata v{NOTEBOOK_VERSION} — initialized")

#### 1.1 Caricamento dataset pulito

In [ ]:
start_year = config["settings"]["start_season"]
end_year   = config["settings"]["end_season"]

start_season    = f"{start_year}-{str(start_year + 1)[-2:]}"
end_season      = f"{end_year}-{str(end_year + 1)[-2:]}"
DATASET_VERSION = f"{start_season}_{end_season}"

PROCESSED_DIR = ROOT / config["data"]["processed_data_dir"]
CLEAN_FILE    = PROCESSED_DIR / f"cleaned_nba_data_{DATASET_VERSION}.csv"

df = pl.read_csv(
    CLEAN_FILE,
    try_parse_dates=True,
    schema_overrides={"game_id": pl.Int64, "team_id": pl.Int64},
)

log.info(f"Loaded  : {CLEAN_FILE.name}")
log.info(f"Shape   : {df.shape[0]:,} rows  x  {df.shape[1]} columns")
log.info(f"Seasons : {df['season'].n_unique()}  ({df['season'].min()} – {df['season'].max()})")
log.info(f"Teams   : {df['team_abbreviation'].n_unique()}")

#### 1.2 Separazione Home / Away e definizione liste variabili

In [ ]:
# ── Separazione home / away per analisi simmetriche ──────────────────────────
df_home = df.filter(pl.col("home_away") == "Home")
df_away = df.filter(pl.col("home_away") == "Away")

# ── Variabili numeriche per dominio ──────────────────────────────────────────
PERF_VARS     = ["pts", "off_rating", "def_rating", "net_rating", "pace", "ts_pct"]
SHOOTING_VARS = ["fgm", "fga", "fg3m", "fg3a", "ftm", "fta"]
ALL_NUM_VARS  = ["pts", "plus_minus", "off_rating", "def_rating", "net_rating",
                 "pace", "ts_pct", "fgm", "fga", "fg3m", "fg3a", "ftm", "fta"]

# ── Target binario ───────────────────────────────────────────────────────────
df = df.with_columns(pl.col("wl").cast(pl.Utf8))
df_w = df.filter(pl.col("wl") == "W")
df_l = df.filter(pl.col("wl") == "L")

log.info(f"Home rows : {df_home.shape[0]:,}")
log.info(f"Away rows : {df_away.shape[0]:,}")
log.info(f"Win  rows : {df_w.shape[0]:,}")
log.info(f"Loss rows : {df_l.shape[0]:,}")

---
### 2. Correlazioni tra Variabili Numeriche

#### 2.1 Heatmap correlazione di Pearson

In [ ]:
corr_matrix = (
    df.select(ALL_NUM_VARS)
    .to_pandas()
    .corr(method="pearson")
)

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    annot_kws={"size": 9},
    ax=ax,
)
ax.set_title("Matrice di correlazione di Pearson — tutte le variabili numeriche", pad=14)
plt.tight_layout()
plt.show()
log.info(f"Coppie con |r| > {HIGH_CORR}: " + str(
    [(c1, c2) for c1 in corr_matrix.columns for c2 in corr_matrix.columns
     if c1 < c2 and abs(corr_matrix.loc[c1, c2]) > HIGH_CORR]
))

#### 2.2 Cluster di variabili altamente correlate

In [ ]:
# ── Clustermap con dendrogramma ───────────────────────────────────────────────
g = sns.clustermap(
    corr_matrix,
    cmap="RdBu_r",
    center=0,
    vmin=-1, vmax=1,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 8},
    linewidths=0.5,
    figsize=(13, 12),
    method="ward",
)
g.figure.suptitle(
    "Clustermap — raggruppamento gerarchico delle variabili (Ward linkage)",
    y=1.01, fontsize=13,
)
plt.show()

#### 2.3 Tabella coppie con alta correlazione (|r| > 0.80)

In [ ]:
high_corr_pairs = []
for i, c1 in enumerate(corr_matrix.columns):
    for c2 in corr_matrix.columns[i + 1:]:
        r = corr_matrix.loc[c1, c2]
        if abs(r) >= HIGH_CORR:
            high_corr_pairs.append({
                "var_1"       : c1,
                "var_2"       : c2,
                "pearson_r"   : round(r, 4),
                "abs_r"       : round(abs(r), 4),
                "implicazione": "Ridondanti — considerare rimozione/PCA" if abs(r) >= 0.95
                                 else "Alta correlazione — monitorare in feature engineering",
            })

high_corr_df = pl.DataFrame(high_corr_pairs).sort("abs_r", descending=True)
log.info(f"Coppie con |r| >= {HIGH_CORR}: {len(high_corr_pairs)}")
show(high_corr_df.to_pandas())

---
### 3. Feature vs Target (W / L)

#### 3.1 Boxplot per ogni variabile numerica separata per W vs L

In [ ]:
n_vars  = len(ALL_NUM_VARS)
n_cols  = 3
n_rows  = (n_vars + n_cols - 1) // n_cols

df_pd = df.select(ALL_NUM_VARS + ["wl"]).to_pandas()

fig, axes = plt.subplots(n_rows, n_cols, figsize=(FIG_W_GRID, n_rows * FIG_H_ROW))
axes = axes.flatten()

for i, var in enumerate(ALL_NUM_VARS):
    sns.boxplot(
        data=df_pd, x="wl", y=var,
        palette={"W": WIN_COLOR, "L": LOSS_COLOR},
        order=["W", "L"],
        width=0.5,
        fliersize=2,
        ax=axes[i],
    )
    axes[i].set_title(var.upper())
    axes[i].set_xlabel("")
    axes[i].set_ylabel(var)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Distribuzione per Win/Loss — tutte le variabili numeriche", y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

#### 3.2 Test statistici Mann-Whitney U per ogni feature

In [ ]:
results = []
w_vals  = df_pd[df_pd["wl"] == "W"]
l_vals  = df_pd[df_pd["wl"] == "L"]

for var in ALL_NUM_VARS:
    w_arr = w_vals[var].dropna().values
    l_arr = l_vals[var].dropna().values

    # Mann-Whitney U (non parametrico, robusto a non-normalità)
    stat_mw, p_mw = mannwhitneyu(w_arr, l_arr, alternative="two-sided")

    # Effect size: rank-biserial correlation r = 1 - 2U / (n1*n2)
    n1, n2    = len(w_arr), len(l_arr)
    effect_r  = 1 - (2 * stat_mw) / (n1 * n2)

    # Differenza delle mediane
    delta_med = float(np.median(w_arr) - np.median(l_arr))

    results.append({
        "variabile"      : var,
        "median_W"       : round(float(np.median(w_arr)), 4),
        "median_L"       : round(float(np.median(l_arr)), 4),
        "delta_median"   : round(delta_med, 4),
        "mw_stat"        : round(stat_mw, 1),
        "p_value"        : round(p_mw, 6),
        "effect_size_r"  : round(effect_r, 4),
        "abs_effect"     : round(abs(effect_r), 4),
        "significativo"  : p_mw < ALPHA,
    })

rank_df = (
    pl.DataFrame(results)
    .sort("abs_effect", descending=True)
    .with_row_index(name="rank", offset=1)
)

show(rank_df.to_pandas())

#### 3.3 Ranking per potere discriminante (|effect size|)

In [ ]:
rank_pd = rank_df.to_pandas().sort_values("abs_effect", ascending=True)

fig, ax = plt.subplots(figsize=(FIG_W_SINGLE, 0.55 * len(ALL_NUM_VARS) + 1))
colors = [WIN_COLOR if s else LOSS_COLOR for s in rank_pd["significativo"]]
bars = ax.barh(rank_pd["variabile"], rank_pd["abs_effect"], color=colors, edgecolor="white")

for bar, val in zip(bars, rank_pd["abs_effect"]):
    ax.text(val + 0.003, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=9)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=WIN_COLOR,  label=f"p < {ALPHA} (significativo)"),
    Patch(facecolor=LOSS_COLOR, label=f"p ≥ {ALPHA} (non significativo)"),
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=9)
ax.set_xlabel("|Effect size r| (rank-biserial)")
ax.set_title("Ranking variabili per potere discriminante su W/L", pad=10)
ax.axvline(0.1, ls="--", color=NEUTRAL, lw=1, label="soglia piccolo (0.1)")
ax.axvline(0.3, ls="--", color="orange", lw=1, label="soglia medio (0.3)")
plt.tight_layout()
plt.show()

---
### 4. Analisi Home vs Away

#### 4.1 Confronto distribuzioni principali per home e away

In [ ]:
KEY_VARS = ["pts", "net_rating", "off_rating", "def_rating", "pace", "ts_pct"]
df_ha    = df.select(KEY_VARS + ["home_away"]).to_pandas()

n_cols = 3
n_rows = (len(KEY_VARS) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(FIG_W_GRID, n_rows * FIG_H_ROW))
axes = axes.flatten()

for i, var in enumerate(KEY_VARS):
    for label, color in [("Home", HOME_COLOR), ("Away", AWAY_COLOR)]:
        subset = df_ha[df_ha["home_away"] == label][var].dropna()
        axes[i].hist(subset, bins=40, alpha=0.55, color=color, label=label, density=True)
        axes[i].axvline(subset.median(), color=color, lw=1.5, ls="--")
    axes[i].set_title(var.upper())
    axes[i].set_xlabel(var)
    axes[i].set_ylabel("Densità")
    axes[i].legend(fontsize=9)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Distribuzioni Home vs Away — variabili chiave", y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

#### 4.2 Quantificazione dell'home advantage

In [ ]:
ha_stats = []
home_pd  = df_ha[df_ha["home_away"] == "Home"]
away_pd  = df_ha[df_ha["home_away"] == "Away"]

for var in KEY_VARS:
    h = home_pd[var].dropna().values
    a = away_pd[var].dropna().values
    stat, p = mannwhitneyu(h, a, alternative="two-sided")
    n1, n2   = len(h), len(a)
    eff      = 1 - (2 * stat) / (n1 * n2)
    ha_stats.append({
        "variabile"    : var,
        "mean_home"    : round(float(np.mean(h)), 3),
        "mean_away"    : round(float(np.mean(a)), 3),
        "delta_mean"   : round(float(np.mean(h) - np.mean(a)), 3),
        "median_home"  : round(float(np.median(h)), 3),
        "median_away"  : round(float(np.median(a)), 3),
        "delta_median" : round(float(np.median(h) - np.median(a)), 3),
        "p_value"      : round(p, 6),
        "effect_r"     : round(eff, 4),
        "significativo": p < ALPHA,
    })

ha_df = pl.DataFrame(ha_stats).sort("effect_r", descending=True)
log.info("Home advantage quantificato su variabili chiave")
show(ha_df.to_pandas())

#### 4.3 Win rate home vs away globale

In [ ]:
df_wl_ha = df.select(["home_away", "wl"]).to_pandas()

win_rate = (
    df_wl_ha
    .groupby("home_away")["wl"]
    .apply(lambda s: (s == "W").mean())
    .reset_index()
    .rename(columns={"wl": "win_rate"})
)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    win_rate["home_away"], win_rate["win_rate"],
    color=[HOME_COLOR, AWAY_COLOR], edgecolor="white", width=0.45,
)
for bar, val in zip(bars, win_rate["win_rate"]):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.005,
            f"{val:.1%}", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.axhline(0.5, ls="--", color=NEUTRAL, lw=1)
ax.set_ylim(0, 0.7)
ax.set_ylabel("Win rate")
ax.set_title("Win rate globale: Home vs Away (2000–2026)")
plt.tight_layout()
plt.show()

#### 4.4 Stabilità dell'home advantage per stagione

In [ ]:
ha_by_season = (
    df.select(["season", "home_away", "wl"])
    .with_columns(pl.col("wl").cast(pl.Utf8))
    .to_pandas()
    .groupby(["season", "home_away"])["wl"]
    .apply(lambda s: (s == "W").mean())
    .reset_index()
    .rename(columns={"wl": "win_rate"})
)

home_wr = ha_by_season[ha_by_season["home_away"] == "Home"].sort_values("season")
away_wr = ha_by_season[ha_by_season["home_away"] == "Away"].sort_values("season")

fig, ax = plt.subplots(figsize=(FIG_W_GRID, 4))
ax.plot(home_wr["season"], home_wr["win_rate"], color=HOME_COLOR, marker="o", ms=4, label="Home")
ax.plot(away_wr["season"], away_wr["win_rate"], color=AWAY_COLOR, marker="o", ms=4, label="Away")
ax.axhline(0.5, ls="--", color=NEUTRAL, lw=1)
ax.fill_between(home_wr["season"], home_wr["win_rate"], away_wr["win_rate"],
                alpha=0.10, color=HOME_COLOR, label="Home advantage gap")

# Evidenzia stagione COVID bubble
covid_season = "2019-20"
if covid_season in home_wr["season"].values:
    idx  = home_wr[home_wr["season"] == covid_season].index[0]
    xpos = home_wr.loc[idx, "season"]
    ax.axvline(xpos, color="red", ls=":", lw=1.5, alpha=0.7)
    ax.text(xpos, ax.get_ylim()[1] * 0.97, " COVID\n bubble",
            color="red", fontsize=8, va="top")

ax.set_xticks(home_wr["season"])
ax.set_xticklabels(home_wr["season"], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Win rate")
ax.set_title("Home advantage per stagione (2000–2026)")
ax.legend()
plt.tight_layout()
plt.show()

---
### 5. Analisi Temporale

#### 5.1 Trend stagionale di NET_RATING, PACE e TS_PCT

In [ ]:
TREND_VARS = ["net_rating", "pace", "ts_pct"]

season_stats = (
    df.select(["season"] + TREND_VARS)
    .group_by("season")
    .agg([
        pl.col(v).mean().alias(f"{v}_mean") for v in TREND_VARS
    ] + [
        pl.col(v).std().alias(f"{v}_std") for v in TREND_VARS
    ])
    .sort("season")
    .to_pandas()
)

fig, axes = plt.subplots(len(TREND_VARS), 1, figsize=(FIG_W_GRID, len(TREND_VARS) * 3.2), sharex=True)

for i, (var, color) in enumerate(zip(TREND_VARS, NBA_PALETTE)):
    mu  = season_stats[f"{var}_mean"]
    sd  = season_stats[f"{var}_std"]
    xs  = season_stats["season"]
    axes[i].plot(xs, mu, color=color, marker="o", ms=4, lw=2, label="media stagionale")
    axes[i].fill_between(xs, mu - sd, mu + sd, alpha=0.15, color=color, label="±1 std")

    # Trend lineare
    x_num = np.arange(len(xs))
    slope, intercept, r, _, _ = stats.linregress(x_num, mu)
    axes[i].plot(xs, intercept + slope * x_num, ls="--", color="gray", lw=1,
                 label=f"trend lineare (slope={slope:.3f})")

    # COVID bubble
    if "2019-20" in xs.values:
        axes[i].axvline("2019-20", color="red", ls=":", lw=1.5, alpha=0.6)

    axes[i].set_ylabel(var.upper())
    axes[i].legend(loc="upper left", fontsize=8)
    axes[i].set_title(f"Trend stagionale — {var.upper()}")

axes[-1].set_xticks(season_stats["season"])
axes[-1].set_xticklabels(season_stats["season"], rotation=45, ha="right", fontsize=8)
fig.suptitle("Evoluzione stagionale delle metriche di gioco (2000–2026)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

#### 5.2 Identificazione stagioni strutturalmente diverse

In [ ]:
# Z-score per stagione sulla media di PACE, TS_PCT, NET_RATING
from scipy.stats import zscore

anomaly_vars = ["pace_mean", "ts_pct_mean", "net_rating_mean"]
ss = season_stats.copy()

for v in anomaly_vars:
    ss[f"{v}_z"] = zscore(ss[v])

# Stagione anomala = z-score assoluto > 2.0 su almeno una metrica
ss["max_abs_z"] = ss[[f"{v}_z" for v in anomaly_vars]].abs().max(axis=1)
anomalous = ss[ss["max_abs_z"] > 2.0][["season"] + [f"{v}_z" for v in anomaly_vars] + ["max_abs_z"]]

log.info(f"Stagioni anomale (|z| > 2 su almeno una metrica): {anomalous['season'].tolist()}")

fig, ax = plt.subplots(figsize=(FIG_W_GRID, 4))
for v, color in zip(anomaly_vars, NBA_PALETTE):
    ax.plot(ss["season"], ss[f"{v}_z"], marker="o", ms=3, lw=1.5, label=v.replace("_mean_z", ""), color=color)
ax.axhline( 2.0, ls="--", color="red",  lw=1, alpha=0.6)
ax.axhline(-2.0, ls="--", color="red",  lw=1, alpha=0.6)
ax.axhline( 0.0, ls="-",  color=NEUTRAL, lw=0.8)

for _, row in anomalous.iterrows():
    ax.axvline(row["season"], color="orange", ls=":", lw=1.5, alpha=0.8)

ax.set_xticks(ss["season"])
ax.set_xticklabels(ss["season"], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Z-score")
ax.set_title("Z-score stagionale — rilevamento stagioni anomale")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

show(anomalous.reset_index(drop=True))

---
### 6. Analisi per Team

#### 6.1 Top / Bottom team per NET_RATING medio (per stagione)

In [ ]:
team_season_stats = (
    df.group_by(["team_abbreviation", "season"])
    .agg([
        pl.col("net_rating").mean().alias("net_rating_mean"),
        (pl.col("wl") == "W").mean().alias("win_rate"),
        pl.len().alias("games"),
    ])
    .sort(["season", "net_rating_mean"], descending=[False, True])
)

# Top 5 e Bottom 5 per NET_RATING medio overall
team_overall = (
    df.group_by("team_abbreviation")
    .agg([
        pl.col("net_rating").mean().alias("net_rating_mean"),
        (pl.col("wl") == "W").mean().alias("win_rate"),
        pl.len().alias("games"),
    ])
    .sort("net_rating_mean", descending=True)
)

top5    = team_overall.head(5)
bottom5 = team_overall.tail(5)
highlight = pl.concat([top5, bottom5]).to_pandas()

fig, ax = plt.subplots(figsize=(10, 6))
colors  = [WIN_COLOR] * 5 + [LOSS_COLOR] * 5
bars    = ax.barh(
    highlight["team_abbreviation"],
    highlight["net_rating_mean"],
    color=colors, edgecolor="white",
)
ax.axvline(0, color=NEUTRAL, lw=1)
for bar, val in zip(bars, highlight["net_rating_mean"]):
    offset = 0.1 if val >= 0 else -0.1
    ha     = "left" if val >= 0 else "right"
    ax.text(val + offset, bar.get_y() + bar.get_height() / 2,
            f"{val:.2f}", va="center", ha=ha, fontsize=9)
ax.set_xlabel("NET_RATING medio (2000–2026)")
ax.set_title("Top 5 e Bottom 5 team per NET_RATING medio (overall)")
plt.tight_layout()
plt.show()

show(team_overall.to_pandas())

#### 6.2 Distribuzione win rate per team

In [ ]:
wr_pd = team_overall.to_pandas().sort_values("win_rate", ascending=False)

fig, ax = plt.subplots(figsize=(FIG_W_GRID, 5))
bar_colors = [WIN_COLOR if w >= 0.5 else LOSS_COLOR for w in wr_pd["win_rate"]]
ax.bar(wr_pd["team_abbreviation"], wr_pd["win_rate"], color=bar_colors, edgecolor="white", width=0.7)
ax.axhline(0.5, ls="--", color=NEUTRAL, lw=1.2)
ax.set_xticklabels(wr_pd["team_abbreviation"], rotation=90, fontsize=8)
ax.set_ylabel("Win rate")
ax.set_title("Win rate per team (aggregato 2000–2026)")
ax.set_ylim(0, 0.8)
plt.tight_layout()
plt.show()

#### 6.3 Stabilità delle performance: autocorrelazione del NET_RATING tra stagioni consecutive

In [ ]:
tss_pd = team_season_stats.to_pandas()

seasons_sorted = sorted(tss_pd["season"].unique())

autocorr_rows = []
for team in tss_pd["team_abbreviation"].unique():
    team_data = tss_pd[tss_pd["team_abbreviation"] == team].set_index("season")["net_rating_mean"]
    team_data = team_data.reindex(seasons_sorted)
    if team_data.notna().sum() >= 5:
        vals     = team_data.dropna().values
        lag1_r   = np.corrcoef(vals[:-1], vals[1:])[0, 1] if len(vals) > 2 else np.nan
        autocorr_rows.append({"team": team, "lag1_autocorr": round(lag1_r, 4)})

autocorr_df = pd.DataFrame(autocorr_rows).dropna().sort_values("lag1_autocorr", ascending=False)
mean_ac      = autocorr_df["lag1_autocorr"].mean()

log.info(f"Autocorrelazione lag-1 media NET_RATING tra team: {mean_ac:.3f}")

fig, ax = plt.subplots(figsize=(FIG_W_GRID, 4))
ax.hist(autocorr_df["lag1_autocorr"], bins=20, color=ACCENT, edgecolor="white")
ax.axvline(mean_ac, color="red", ls="--", lw=1.5, label=f"media = {mean_ac:.3f}")
ax.axvline(0, color=NEUTRAL, lw=1)
ax.set_xlabel("Autocorrelazione lag-1 (NET_RATING stagionale)")
ax.set_ylabel("Numero team")
ax.set_title("Stabilità delle performance: autocorrelazione NET_RATING tra stagioni consecutive")
ax.legend()
plt.tight_layout()
plt.show()

# Heatmap team x season
pivot = tss_pd.pivot(index="team_abbreviation", columns="season", values="net_rating_mean")
fig, ax = plt.subplots(figsize=(FIG_W_GRID + 4, 12))
sns.heatmap(
    pivot, cmap="RdBu_r", center=0, linewidths=0.3,
    annot=False, ax=ax, cbar_kws={"label": "NET_RATING medio"},
)
ax.set_title("NET_RATING medio per team × stagione", pad=10)
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

---
### 7. Multicollinearità

#### 7.1 Variance Inflation Factor (VIF)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools import add_constant

vif_data = df.select(ALL_NUM_VARS).drop_nulls().to_pandas()
X_const  = add_constant(vif_data)

vif_results = []
for i, col in enumerate(X_const.columns[1:], start=1):  # skip constant
    vif_val = variance_inflation_factor(X_const.values, i)
    vif_results.append({
        "variabile"     : col,
        "VIF"           : round(vif_val, 2),
        "problematica"  : vif_val > VIF_THRESHOLD,
        "livello"       : "critico (>10)" if vif_val > 10
                          else "alto (5–10)" if vif_val > VIF_THRESHOLD
                          else "ok (<5)",
    })

vif_df = pl.DataFrame(vif_results).sort("VIF", descending=True)
show(vif_df.to_pandas())

# Visualizzazione
vif_pd = vif_df.to_pandas().sort_values("VIF", ascending=True)
fig, ax = plt.subplots(figsize=(8, 0.55 * len(ALL_NUM_VARS) + 1))
bar_c = [LOSS_COLOR if v > VIF_THRESHOLD else WIN_COLOR for v in vif_pd["VIF"]]
bars  = ax.barh(vif_pd["variabile"], vif_pd["VIF"], color=bar_c, edgecolor="white")
for bar, val in zip(bars, vif_pd["VIF"]):
    ax.text(val + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}", va="center", fontsize=9)
ax.axvline(VIF_THRESHOLD, ls="--", color="orange", lw=1.5, label=f"soglia VIF = {VIF_THRESHOLD}")
ax.axvline(10, ls="--", color="red", lw=1.5, label="soglia critica = 10")
ax.set_xlabel("VIF")
ax.set_title("Variance Inflation Factor — tutte le variabili candidate")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

#### 7.2 Scatter matrix tra variabili predittive principali

In [ ]:
SCATTER_VARS = ["pts", "net_rating", "off_rating", "def_rating", "pace", "ts_pct", "plus_minus"]

scatter_pd = df.select(SCATTER_VARS + ["wl"]).to_pandas().dropna()

g = sns.pairplot(
    scatter_pd,
    vars=SCATTER_VARS,
    hue="wl",
    palette={"W": WIN_COLOR, "L": LOSS_COLOR},
    plot_kws={"alpha": 0.25, "s": 5},
    diag_kind="kde",
    corner=True,
)
g.figure.suptitle("Scatter matrix — variabili predittive principali colorata per W/L",
                  y=1.01, fontsize=13)
plt.show()

#### 7.3 Documentazione coppie problematiche

In [ ]:
# Combina informazioni da correlazione e VIF
problematic_vars = set(
    vif_df.filter(pl.col("problematica").cast(pl.Boolean))["variabile"].to_list()
)

prob_pairs_rows = []
for row in high_corr_pairs:
    v1, v2 = row["var_1"], row["var_2"]
    prob_pairs_rows.append({
        "var_1"         : v1,
        "var_2"         : v2,
        "pearson_r"     : row["pearson_r"],
        "v1_vif_high"   : v1 in problematic_vars,
        "v2_vif_high"   : v2 in problematic_vars,
        "azione_suggerita": (
            "Rimuovere una delle due o applicare PCA/combinazione lineare"
            if abs(row["pearson_r"]) >= 0.95
            else "Monitorare; valutare rimozione in base all'importanza nel modello"
        ),
    })

if prob_pairs_rows:
    prob_df = pl.DataFrame(prob_pairs_rows).sort("pearson_r", descending=True)
    log.info(f"Coppie problematiche documentate: {len(prob_pairs_rows)}")
    show(prob_df.to_pandas())
else:
    log.info("Nessuna coppia con |r| >= 0.80 identificata.")
    print("Nessuna coppia con |r| >= 0.80 identificata.")

---
### 8. Sintesi e Ipotesi per la Modellazione

#### 8.1 Variabili con maggiore potere predittivo atteso

In [ ]:
# Unione delle evidenze raccolte nei capitoli precedenti
rank_pd_sorted = rank_df.sort("abs_effect", descending=True).to_pandas()
vif_pd_map     = {r["variabile"]: r["VIF"] for r in vif_df.to_dicts()}

summary_rows = []
for _, row in rank_pd_sorted.iterrows():
    var     = row["variabile"]
    vif_val = vif_pd_map.get(var, float("nan"))
    summary_rows.append({
        "variabile"          : var,
        "rank_discriminante" : int(row["rank"]),
        "abs_effect_r"       : row["abs_effect"],
        "p_value"            : row["p_value"],
        "significativo"      : row["significativo"],
        "VIF"                : round(vif_val, 2),
        "VIF_ok"             : vif_val <= VIF_THRESHOLD,
        "raccomandazione"    : (
            "INCLUDI — forte segnale, bassa multicollinearità"
            if row["abs_effect"] >= 0.3 and vif_val <= VIF_THRESHOLD
            else "INCLUDI CON CAUTELA — forte segnale, alta multicollinearità"
            if row["abs_effect"] >= 0.3 and vif_val > VIF_THRESHOLD
            else "VALUTA — segnale moderato"
            if row["abs_effect"] >= 0.1
            else "ESCLUDI — segnale debole"
        ),
    })

summary_df = pl.DataFrame(summary_rows)
show(summary_df.to_pandas())

#### 8.2 Ipotesi formali da verificare in modellazione

In [ ]:
hypotheses = [
    {
        "id"         : "H1",
        "ipotesi"    : "net_rating è il predittore singolo più forte per W/L",
        "evidenza"   : "Effect size r tra i più alti nel ranking Mann-Whitney",
        "da_testare" : "Confronto importanza feature in modelli albero e logistic regression",
    },
    {
        "id"         : "H2",
        "ipotesi"    : "off_rating e def_rating sono ridondanti rispetto a net_rating",
        "evidenza"   : "net_rating = off_rating − def_rating (per definizione), alta correlazione attesa",
        "da_testare" : "Confronto modello con/senza net_rating vs off+def separati",
    },
    {
        "id"         : "H3",
        "ipotesi"    : "L'home advantage è statisticamente significativo ma si è ridotto nel tempo",
        "evidenza"   : "Win rate home > 0.5 ma con possibile trend decrescente; anomalia COVID bubble",
        "da_testare" : "Regressione win rate home ~ stagione; t-test pre/post 2019",
    },
    {
        "id"         : "H4",
        "ipotesi"    : "La stagione 2019-20 (COVID bubble) è un outlier strutturale da gestire",
        "evidenza"   : "Z-score anomalo su PACE e win rate home nella stagione bubble",
        "da_testare" : "Confronto performance modello con/senza stagione 2019-20",
    },
    {
        "id"         : "H5",
        "ipotesi"    : "Il PACE e TS_PCT mostrano trend strutturali che richiedono normalizzazione per stagione",
        "evidenza"   : "Trend lineare visibile nelle serie temporali",
        "da_testare" : "Feature engineering: z-score within-season per PACE e TS_PCT",
    },
    {
        "id"         : "H6",
        "ipotesi"    : "Le metriche di tiro (fgm, fga, fg3m, fg3a, ftm, fta) sono ridondanti rispetto a ts_pct",
        "evidenza"   : "TS_PCT è una sintesi efficiente del volume + efficienza; alta correlazione reciproca attesa",
        "da_testare" : "Analisi importanza in modelli + test con/senza variabili raw shooting",
    },
    {
        "id"         : "H7",
        "ipotesi"    : "La performance del team nella stagione corrente è predittiva della prossima (stabilità)",
        "evidenza"   : "Autocorrelazione lag-1 media NET_RATING positiva tra team",
        "da_testare" : "Feature lagged: net_rating_rolling, win_rate_prev_season",
    },
]

hyp_df = pl.DataFrame(hypotheses)
log.info(f"Ipotesi formalizzate: {len(hypotheses)}")
show(hyp_df.to_pandas())

#### 8.3 Lista variabili da escludere o aggregare

In [ ]:
exclusion_plan = [
    {
        "variabile"  : "off_rating",
        "motivo"     : "Ridondante con net_rating (def: net = off − def). Alta multicollinearità attesa.",
        "azione"     : "Escludere se net_rating incluso; oppure usare come feature separata senza net_rating",
    },
    {
        "variabile"  : "def_rating",
        "motivo"     : "Come off_rating: componente di net_rating.",
        "azione"     : "Escludere se net_rating incluso; utile per modelli che distinguono attacco/difesa",
    },
    {
        "variabile"  : "plus_minus",
        "motivo"     : "Proxy diretto del risultato finale (correlazione quasi perfetta con wl).",
        "azione"     : "Escludere come feature: data leakage — è determinato dal risultato della partita",
    },
    {
        "variabile"  : "fgm / fga / fg3m / fg3a / ftm / fta",
        "motivo"     : "Metriche raw ridondanti con ts_pct e pts; alta correlazione reciproca.",
        "azione"     : "Aggregare in ratios (fg_pct, fg3_pct, ft_pct) oppure sostituire con ts_pct",
    },
    {
        "variabile"  : "game_id / team_id",
        "motivo"     : "Identificatori tecnici, nessun segnale predittivo.",
        "azione"     : "Escludere come feature",
    },
    {
        "variabile"  : "is_outlier_flag",
        "motivo"     : "Flag di qualità dati, non feature predittiva.",
        "azione"     : "Usare solo come filtro; escludere dal training set o gestire separatamente",
    },
]

excl_df = pl.DataFrame(exclusion_plan)
log.info(f"Variabili da escludere/aggregare: {len(exclusion_plan)}")
show(excl_df.to_pandas())

#### 8.4 Riepilogo — Feature shortlist per il notebook di Feature Engineering

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║          FEATURE SHORTLIST — punto di partenza per Feature Engineering      ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  INCLUDI DIRETTAMENTE                                                        ║
║    • net_rating      — predittore più forte, bassa ridondanza con ts_pct     ║
║    • pts             — forte segnale, interpretabile                          ║
║    • ts_pct          — efficienza sintetica, trend temporale → z-score       ║
║    • pace            — strutturale, trend temporale → z-score within-season  ║
║    • home_away       — home advantage significativo (~60% win rate home)      ║
║                                                                              ║
║  INCLUDI CON TRASFORMAZIONE                                                  ║
║    • fg_pct          = fgm / fga  (ratio al posto delle raw)                 ║
║    • fg3_pct         = fg3m / fg3a                                            ║
║    • ft_pct          = ftm / fta                                              ║
║    • season (z-norm) — per catturare drift strutturale di regole NBA          ║
║                                                                              ║
║  INCLUDI COME FEATURE LAGGED (Feature Engineering)                           ║
║    • net_rating_rolling_N   — media mobile N partite precedenti              ║
║    • win_rate_prev_season   — stabilità inter-stagionale                      ║
║                                                                              ║
║  ESCLUDI                                                                     ║
║    • plus_minus      — data leakage (determinato dal risultato)               ║
║    • off_rating      — ridondante se net_rating incluso                       ║
║    • def_rating      — idem                                                   ║
║    • fgm,fga,fg3m,fg3a,ftm,fta — sostituiti da ratios                        ║
║    • game_id,team_id — identificatori tecnici                                 ║
║    • is_outlier_flag — flag qualità, non predittore                           ║
║                                                                              ║
║  STAGIONI ANOMALE DA GESTIRE                                                  ║
║    • 2019-20 (COVID bubble) — testare esclusione o dummy flag                 ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

log.info("EDA Bivariata completata. Notebook pronto per Feature Engineering.")